# RFSoC XM655 human-readable server notebook

This notebook intentionally contains only the high-level flow. Most implementation details are in `rfsoc_notebook_server.py` so you can inspect the server logic in one normal Python file.

Raw TCP streaming is kept compatible with the working notebook: MATLAB commands request raw `int16` frames, and the notebook sends the same frame format back. Notebook DAC beamforming/tracking is added as an extra command layer, not as a different data format.


## 1. Edit runtime settings

Start with `RAW_ONLY = True` when diagnosing speed. Then enable the algorithm, and only after that enable DAC tracking. This isolates where slowdown appears.


In [ ]:
# TCP settings
SERVER_PORT = 4000
DEFAULT_STREAM_FRAMES = 1     # Keep 1 for old MATLAB request/response mode. MATLAB can still send stream N.

# Diagnostic switches
RAW_ONLY = False              # True = raw TCP only; disables algorithm + notebook DAC tracking.
ENABLE_ALGORITHM_WORKER = True
ENABLE_DAC_TRACKING = True

# Logging
LOG_LEVEL = "summary"          # "quiet", "summary", or "debug"
TIMING_EVERY = 200             # Only used when LOG_LEVEL = "debug"
BURST_SUMMARY_EVERY = 50       # 0 disables burst summary logs
PRINT_COMMANDS = False

# CPU allocation. Use None to disable pinning.
ALGO_CPU_CORE = 1
SERVER_CPU_CORES = [0, 2, 3]

# Algorithm settings
ALGO_QUEUE_SIZE = 1            # newest-frame behavior; do not let algorithm block raw streaming
ALGO_EVERY_N_FRAMES = 1
ALGO_CONFIG_PATH = "algo_config.npz"  # Initial config; can be replaced by MVDR reload commands
ALGO_PRINT_EVERY = 200

# Manual MATLAB MVDR corrections. Upload these .mat files through Jupyter GUI into
# the same folder as this notebook. They are automatically reloaded when MATLAB
# sends dacbf tracking / dacbf retracking.
MVDR_AUTO_RELOAD_ON_TRACKING = True
MVDR_MODE = "MVDRman_corr"              # "MVDRman" = identity, "MVDRman_corr" = load .mat files
MVDR_STEERING_FILE = "steering_correction.mat"
MVDR_POWER_FILE = "pow_claibration_intrp.mat"
MVDR_SCAN_START_DEG = -60.0
MVDR_SCAN_STOP_DEG = 60.0
MVDR_SCAN_STEP_DEG = 1.0

# DAC BF tracking settings
DAC_TRACKING_MIN_UPDATE_INTERVAL_S = 0.05   # 20 Hz max RFDC phase writes by default
DAC_TRACKING_MIN_ANGLE_STEP_DEG = 0.25
DAC_TRACKING_PHASE_SIGN = -1.0              # use +1 if TX direction is reversed
DAC_TRACKING_PRINT_EVERY = 200


## 2. Import code and build the config

All TCP, worker, and DAC-BF logic is now in `rfsoc_notebook_server.py`.


In [ ]:
%load_ext autoreload
%autoreload 2

from rfsoc_notebook_server import ServerConfig, RfsocNotebookServer
from mts import doaMtsOverlay

config = ServerConfig(
    port=SERVER_PORT,
    default_stream_frames=DEFAULT_STREAM_FRAMES,
    raw_only=RAW_ONLY,
    enable_algorithm_worker=ENABLE_ALGORITHM_WORKER,
    enable_dac_tracking=ENABLE_DAC_TRACKING,
    log_level=LOG_LEVEL,
    timing_every=TIMING_EVERY,
    burst_summary_every=BURST_SUMMARY_EVERY,
    print_commands=PRINT_COMMANDS,
    server_cpu_cores=SERVER_CPU_CORES,
    algo_cpu_core=ALGO_CPU_CORE,
    algo_queue_size=ALGO_QUEUE_SIZE,
    algo_every_n_frames=ALGO_EVERY_N_FRAMES,
    algo_config_path=ALGO_CONFIG_PATH,
    algo_print_every=ALGO_PRINT_EVERY,
    mvdr_auto_reload_on_tracking=MVDR_AUTO_RELOAD_ON_TRACKING,
    mvdr_mode=MVDR_MODE,
    mvdr_steering_file=MVDR_STEERING_FILE,
    mvdr_power_file=MVDR_POWER_FILE,
    mvdr_scan_start_deg=MVDR_SCAN_START_DEG,
    mvdr_scan_stop_deg=MVDR_SCAN_STOP_DEG,
    mvdr_scan_step_deg=MVDR_SCAN_STEP_DEG,
    dac_tracking_min_update_interval_s=DAC_TRACKING_MIN_UPDATE_INTERVAL_S,
    dac_tracking_min_angle_step_deg=DAC_TRACKING_MIN_ANGLE_STEP_DEG,
    dac_tracking_phase_sign=DAC_TRACKING_PHASE_SIGN,
    dac_tracking_print_every=DAC_TRACKING_PRINT_EVERY,
)

config


## 3. Load overlay and prepare the server

The algorithm worker is started before the server begins accepting MATLAB connections. The overlay is still loaded in the main notebook process.


In [ ]:
server = RfsocNotebookServer(config)

# Load RFSoC overlay. Restart the kernel before re-running this cell after code changes.
overlay = doaMtsOverlay("mts.bit")

# Attach overlay, start worker if enabled, and open listening socket.
server.prepare(overlay)

server.status()


## 4. Run the TCP server

This cell blocks while MATLAB is connected. Stop it with the notebook stop button or `Kernel → Interrupt`. Cleanup runs automatically.

MATLAB/notebook DAC-BF commands supported:

```text
dacbf off
dacbf tracking
dacbf retracking
dacbfcorr 0/0/-600/12300
dacbfcorrdeg 0/0/-6/123
dacbfsign -1
mvdrreload
mvdrscan -60/60/1
mvdrmode MVDRman_corr
mvdrfiles steering_correction.mat/pow_claibration_intrp.mat
```

When MATLAB switches to `dacbf tracking` or `dacbf retracking`, the server reloads the latest manual MVDR correction files automatically.


In [ ]:
server.run_forever()


## 5. Optional manual cleanup/status

Use this only if you interrupted during setup or want to inspect counters after the server returns.


In [ ]:
server.status()
# server.close()
